# CNN Step-by-Step Demo: Handwritten Digit Classification

## Learning objective
Build a simple **Convolutional Neural Network (CNN)** to recognize handwritten digits (0–9) using MNIST.

### CNN pipeline
`Image → Convolution → ReLU → Pooling → Convolution → Pooling → Flatten → Dense → Prediction`

Each code block is followed by notes explaining what it does and why it is needed.


## 1. Install the required libraries

Run this once in Anaconda Prompt / Terminal if needed:

```bash
pip install tensorflow matplotlib numpy
```

If these packages are already installed, skip this step.


## 2. Import libraries

We need TensorFlow for deep learning, Matplotlib for images/graphs, and NumPy for array operations.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)


### Notes

- `tensorflow` provides the deep-learning framework and Keras API.
- `matplotlib.pyplot` displays images and graphs.
- `numpy` handles numerical arrays.
- `tf.__version__` checks the installed TensorFlow version.


## 3. Load the MNIST dataset

MNIST contains 70,000 grayscale handwritten-digit images:
- 60,000 for training
- 10,000 for testing
- Each image is 28 × 28 pixels


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)


### Notes

- `x_train` = training images.
- `y_train` = correct digit labels.
- `x_test` = unseen test images.
- `y_test` = correct labels for test images.

Expected shapes:

```text
x_train → (60000, 28, 28)
y_train → (60000,)
x_test  → (10000, 28, 28)
y_test  → (10000,)
```

**Teaching point:** An image is represented as a 28 × 28 matrix of pixel values.


## 4. Display some training images

Let's look at the actual data before building the CNN.

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.show()


### Notes

- `plt.figure()` creates the drawing area.
- `plt.subplot(2, 5, i + 1)` creates a 2 × 5 grid.
- `plt.imshow()` displays an image.
- `cmap="gray"` uses grayscale.
- `y_train[i]` is the actual digit label.



## 5. Normalize pixel values

Original pixels range from **0 to 255**. We convert them to **0 to 1**.


In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())


### Notes

Dividing by 255 converts:

- `0 → 0.0`
- `255 → 1.0`

This is called **normalization** and generally makes neural-network training easier.


## 6. Add the channel dimension

CNNs normally expect:

`Height × Width × Channels`

MNIST is grayscale, so it has one channel. We convert `28 × 28` into `28 × 28 × 1`.


In [ ]:
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("New training shape:", x_train.shape)
print("New test shape:", x_test.shape)


### Notes

`np.newaxis` adds one dimension.

Before:

`60000 × 28 × 28`

After:

`60000 × 28 × 28 × 1`

The final `1` means one grayscale channel. RGB images normally have 3 channels.


# 7. Build the CNN

### Architecture

```text
28 × 28 × 1 image
       ↓
Conv2D: 32 filters
       ↓
MaxPooling
       ↓
Conv2D: 64 filters
       ↓
MaxPooling
       ↓
Flatten
       ↓
Dense: 128 neurons
       ↓
Dense: 10 outputs
       ↓
Digit 0–9
```


In [ ]:
model = tf.keras.Sequential([
    # First convolution layer
    tf.keras.layers.Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    ),

    # First pooling layer
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Second convolution layer
    tf.keras.layers.Conv2D(
        64,
        kernel_size=(3, 3),
        activation="relu"
    ),

    # Second pooling layer
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Convert feature maps into a one-dimensional vector
    tf.keras.layers.Flatten(),

    # Fully connected layer
    tf.keras.layers.Dense(128, activation="relu"),

    # Output layer: one output for each digit 0–9
    tf.keras.layers.Dense(10, activation="softmax")
])


## Understanding each CNN block

### `Conv2D(32, kernel_size=(3,3))`
Learns small visual patterns using 32 different 3 × 3 filters. Early filters may learn edges, curves, or simple shapes.

### `activation="relu"`
ReLU means **Rectified Linear Unit**. It keeps positive values and changes negative values to zero. It gives the network the ability to learn non-linear patterns.

### `MaxPooling2D((2,2))`
Reduces the spatial size of feature maps while retaining strong activations. This reduces computation.

### Second `Conv2D`
Learns more complex combinations of features from the earlier layer.

A useful classroom explanation is:

`Edges → curves → shapes → digit patterns`

### `Flatten()`
Converts the 2D feature maps into one long vector for the dense layers.

### `Dense(128)`
A fully connected layer that combines learned features for classification.

### `Dense(10, softmax)`
There are 10 classes: digits 0 through 9. Softmax converts the outputs into probabilities.


## 8. Display the model architecture

In [ ]:
model.summary()


### Teaching point

`model.summary()` shows the layers and trainable parameters.

> A CNN is not one magical image-recognition operation. It is a sequence of layers, with each layer transforming the image representation.


# 9. Compile the model

Before training, we specify how the model should learn.


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


### Notes

**Optimizer – `adam`**  
Adjusts the model's weights during training to reduce error.

**Loss – `sparse_categorical_crossentropy`**  
Measures how far predictions are from the correct digit labels.

**Metric – `accuracy`**  
Measures the percentage of correct predictions.


# 10. Train the CNN

For a classroom demonstration, 5 epochs are enough to show the learning process.


In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


### Notes

- `x_train` = training images.
- `y_train` = correct answers.
- `epochs=5` = the model sees the training data five times.
- `batch_size=64` = processes 64 images before a weight update.
- `validation_split=0.1` = reserves 10% of training data for validation.

**Key idea:** The CNN repeatedly predicts, calculates error, and adjusts its weights.


## 11. Visualize training accuracy

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training Accuracy")
plt.legend()
plt.show()


### Teaching point

If accuracy generally increases, the model is learning useful patterns. The goal here is understanding the learning process, not simply chasing the highest accuracy.


# 12. Evaluate the CNN on test data

The test set contains images that were not used for training.


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", test_accuracy)


### Why test accuracy matters

Training accuracy tells us performance on training data. Test accuracy gives us an idea of how well the model generalizes to unseen images.

A simple CNN like this will often reach around **98–99% test accuracy**, though results vary.


# 13. Make a prediction for one image

Now give the CNN one test image and ask it to predict the digit.


In [ ]:
image = x_test[0]

prediction = model.predict(image[np.newaxis, ...])

predicted_digit = np.argmax(prediction)

print("Predicted digit:", predicted_digit)
print("Actual digit:", y_test[0])


### Notes

- `x_test[0]` selects one test image.
- `np.newaxis` adds the batch dimension expected by Keras.
- `model.predict()` asks the trained CNN for a prediction.
- `np.argmax()` finds the class with the highest probability.

**Key idea:** The CNN receives pixel values and produces probabilities for possible digits.


# 14. Display the image and prediction

In [ ]:
plt.imshow(x_test[0].squeeze(), cmap="gray")
plt.title(f"Predicted: {predicted_digit} | Actual: {y_test[0]}")
plt.axis("off")
plt.show()


### Notes

`squeeze()` removes the extra channel dimension for convenient display.

- **Predicted** = what the CNN thinks.
- **Actual** = the correct dataset label.


# 15. See the probability for every digit

Softmax gives a probability distribution across the 10 possible classes.


In [ ]:
probabilities = prediction[0]

for digit, probability in enumerate(probabilities):
    print(f"Digit {digit}: {probability * 100:.2f}%")


### Teaching point

The model does not simply say `"This is 7."`

It produces probabilities for all ten classes. The class with the highest probability becomes the prediction.


# 16. Test 10 images at once


In [ ]:
predictions = model.predict(x_test[:10])

plt.figure(figsize=(12, 5))

for i in range(10):
    predicted = np.argmax(predictions[i])

    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title(f"Pred: {predicted}\nActual: {y_test[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()


# 17. The CNN idea in one picture

```text
                IMAGE
              28 × 28 × 1
                   │
                   ▼
           ┌───────────────┐
           │  Convolution  │
           │ Find features │
           └───────┬───────┘
                   │
                   ▼
                 ReLU
                   │
                   ▼
           ┌───────────────┐
           │    Pooling    │
           │  Reduce size  │
           └───────┬───────┘
                   │
                   ▼
            More Convolution
                   │
                   ▼
                Flatten
                   │
                   ▼
             Dense Layer
                   │
                   ▼
          0 1 2 3 4 5 6 7 8 9
                   │
                   ▼
              Prediction
```

### Simple explanation

**Edges → curves → shapes → digit patterns → final classification**

A CNN learns these useful visual representations automatically from training examples.


# 18. Classroom challenge: CNN vs Dense network

a quick question :

> What happens if we flatten the image immediately and use only fully connected layers?

The following model lets you compare the idea.


In [ ]:
dense_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

dense_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

dense_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


### Discussion

**Dense network**

`Image → Flatten → Dense → Prediction`

**CNN**

`Image → Convolution → Pooling → Features → Dense → Prediction`

CNNs are powerful for images because convolution layers preserve and exploit local spatial relationships. The same learned filter can detect a useful pattern at different locations.


# 19. Final takeaway for students

## CNN = Convolutional Neural Network

A CNN is a neural-network architecture designed especially for data with spatial structure, particularly images.

### Remember these four ideas

1. **Convolution** → learns visual features.
2. **ReLU** → introduces non-linearity.
3. **Pooling** → reduces spatial size while keeping important information.
4. **Dense + Softmax** → uses learned features to classify the image.

### One-line definition

> **A CNN learns useful visual features from images and combines those features to make predictions.**

### Real-world applications

- Face recognition
- Medical image analysis
- Defect detection in manufacturing
- Number-plate recognition
- Object detection
- Satellite image analysis
- Autonomous vehicles
- Quality inspection
- Handwriting recognition
